# NI USB-6002 Data Logger (Live View + CSV Export)

This notebook streams analog input data from an **NI USB-6002** using **`nidaqmx`**, shows a **live plot**, and exports to **CSV**.

## Student workflow
1. Run the setup cells (backend, imports, device detection).
2. Edit the **Settings** cell (channels, sample rate, duration, terminal mode).
3. Run **Start logging (live plot)**.
4. Run **Export CSV**.

> If you only change one thing, change the **Settings** cell.


In [ ]:
%matplotlib widget

## Imports

If you get `ModuleNotFoundError: nidaqmx`, install it in your active environment:
```bash
pip install nidaqmx
```
You also need NI-DAQmx installed on the computer (NI drivers).


In [ ]:
import time
import threading
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nidaqmx
from nidaqmx.constants import AcquisitionType, TerminalConfiguration

## Detect your NI device

This prints connected NI devices. For the USB-6002, the device name is often `Dev1`.


In [ ]:
system = nidaqmx.system.System.local()
devices = [d.name for d in system.devices]
print("NI devices detected:", devices)

if not devices:
    print("\nNo devices found. Troubleshooting:")
    print("• Confirm the USB-6002 is plugged in.")
    print("• Open NI MAX and verify the device is visible.")
    print("• Confirm NI-DAQmx drivers are installed.")


## Settings (edit this)

**Channels:** list the analog input channels you want, like `["ai0"]` or `["ai0","ai1"]`.

**Terminal config:**  
- `TerminalConfiguration.RSE` (single-ended, referenced to AI GND)  
- `TerminalConfiguration.DIFFERENTIAL` (differential measurement)

> The USB-6002 supports differential on certain channel pairs; check NI MAX / device manual if unsure.


In [ ]:
# ---------- Hardware ----------
DEVICE = devices[0] if devices else "Dev1"          # e.g., "Dev1"
CHANNELS = ["ai0"]                                   # e.g., ["ai0","ai1"]
PHYS_CHANS = [f"{DEVICE}/{ch}" for ch in CHANNELS]

TERM_CFG = TerminalConfiguration.RSE                 # or TerminalConfiguration.DIFFERENTIAL
MIN_V, MAX_V = -10.0, 10.0                           # input range (V)

# ---------- Sampling ----------
SAMPLE_RATE_HZ = 1000.0                              # Hz
DURATION_S = 10.0                                    # seconds
CHUNK_SAMPLES = 200                                  # read this many samples each loop

# ---------- Live plot window ----------
PLOT_WINDOW_S = 5.0                                  # show the last N seconds in the live plot


## Start logging (live plot)

This cell:
- starts continuous acquisition
- reads data in a background thread
- updates the plot using `FuncAnimation`
- stops automatically after `DURATION_S`

When it finishes, your full dataset is stored in `log_time_s` and `log_data` for export.


In [ ]:
from matplotlib.animation import FuncAnimation

# --------- Buffers for plotting (last PLOT_WINDOW_S) ---------
plot_time = deque(maxlen=int(PLOT_WINDOW_S * SAMPLE_RATE_HZ) + 1)
plot_data = [deque(maxlen=int(PLOT_WINDOW_S * SAMPLE_RATE_HZ) + 1) for _ in CHANNELS]

# --------- Full logs for CSV export ---------
log_time_s = []                 # list[float]
log_data = [[] for _ in CHANNELS]  # list[list[float]]

stop_event = threading.Event()

def reader_thread_fn(task, t0):
    """Read chunks continuously and append to buffers."""
    while not stop_event.is_set():
        # Read shape: list[n_channels][n_samples] when multiple channels
        data = task.read(number_of_samples_per_channel=CHUNK_SAMPLES, timeout=2.0)
        now = time.perf_counter()
        dt = 1.0 / SAMPLE_RATE_HZ

        # Normalize to 2D list: channels x samples
        if len(CHANNELS) == 1:
            data = [data]

        # Build time stamps for this chunk (assumes uniform sampling)
        # End time is 'now - t0' (approx). We back-calculate the chunk times.
        t_end = now - t0
        t_start = t_end - (len(data[0]) - 1) * dt
        t_chunk = t_start + np.arange(len(data[0])) * dt

        # Append to plot buffers and full logs
        for ti in t_chunk:
            plot_time.append(ti)
            log_time_s.append(ti)

        for ci, ch_samples in enumerate(data):
            plot_data[ci].extend(ch_samples)
            log_data[ci].extend(ch_samples)

def build_task():
    task = nidaqmx.Task()
    for phys in PHYS_CHANS:
        task.ai_channels.add_ai_voltage_chan(
            physical_channel=phys,
            min_val=MIN_V,
            max_val=MAX_V,
            terminal_config=TERM_CFG
        )
    task.timing.cfg_samp_clk_timing(
        rate=SAMPLE_RATE_HZ,
        sample_mode=AcquisitionType.CONTINUOUS,
        samps_per_chan=CHUNK_SAMPLES  # buffer hint
    )
    return task

# --------- Plot setup ---------
plt.close("all")
fig, ax = plt.subplots(figsize=(8, 4))
ax.set_title("Live NI USB-6002 Stream")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (V)")
ax.grid(True)

lines = []
for i, ch in enumerate(CHANNELS):
    (ln,) = ax.plot([], [], label=ch)
    lines.append(ln)
ax.legend(loc="upper right")
plt.show()

# --------- Start acquisition ---------
task = build_task()
t0 = time.perf_counter()
task.start()

thread = threading.Thread(target=reader_thread_fn, args=(task, t0), daemon=True)
thread.start()

# --------- Animation update ---------
def update(_frame):
    if len(plot_time) < 2:
        return lines

    t = np.asarray(plot_time)

    for i, ln in enumerate(lines):
        y = np.asarray(plot_data[i])
        ln.set_data(t, y)

    ax.set_xlim(t.min(), t.max())
    # autoscale y based on visible data (with a small margin)
    y_all = np.concatenate([np.asarray(d) for d in plot_data if len(d) > 0])
    if y_all.size:
        y_min, y_max = float(y_all.min()), float(y_all.max())
        pad = 0.05 * (y_max - y_min + 1e-9)
        ax.set_ylim(y_min - pad, y_max + pad)

    return lines

ani = FuncAnimation(fig, update, interval=50, blit=False)

# --------- Run for DURATION_S, then stop ---------
time.sleep(DURATION_S)

stop_event.set()
thread.join(timeout=2.0)

task.stop()
task.close()

print(f"Done. Logged {len(log_time_s)} samples per channel (approx).")

## Export to CSV

This creates a CSV with columns:
- `time_s`
- one column per channel (e.g., `ai0`, `ai1`)

The file is saved in the current working directory.


In [ ]:
# Build dataframe
df = pd.DataFrame({"time_s": log_time_s})
for i, ch in enumerate(CHANNELS):
    df[ch] = log_data[i]

# Basic sanity check
print(df.head())
print("\nRows:", len(df), "Columns:", len(df.columns))

# Save
timestamp = time.strftime("%Y%m%d_%H%M%S")
csv_name = f"ni_usb6002_log_{timestamp}.csv"
df.to_csv(csv_name, index=False)
print("\nSaved:", csv_name)

## Troubleshooting

### No device found
- Verify the USB-6002 shows up in **NI MAX**.
- Re-plug the USB cable.
- Install/repair **NI-DAQmx**.

### Terminal configuration issues (RSE vs Differential)
- If readings look wrong or saturated, confirm the wiring matches `TERM_CFG`.
- For differential, confirm you are using a valid pair and correct terminals.

### Plot not updating during the run
- Make sure you ran `%matplotlib widget` and you are using the correct Jupyter kernel.
- Avoid `time.sleep()` loops for plot updates. This notebook uses `FuncAnimation` (recommended).
